# Draft INQUIRE fine-tuning examples (Component 3)

Runs `draft_inquire_dataset.py` on Colab with Gemini through Colab's built-in models and distilabel.
Outputs are **drafts**: every row is reviewed by the owner before it becomes a seed example.

Order matters: **scenarios** (freezes the 10 test scenarios) → **replies** → **drafts**.
Each stage is resumable: if one fails part-way, re-run the same cell.
Outputs go to Google Drive so a runtime disconnect loses nothing.

In [ ]:
!pip -q install "distilabel[openai]==1.5.3"

Upload `draft_inquire_dataset.py` and `data/educator_seed.jsonl` from `component3_socratic_educator/`.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert {"draft_inquire_dataset.py", "educator_seed.jsonl"} <= set(uploaded), sorted(uploaded)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/c3_inquire_drafts"

Check model access. This also loads the Colab model-proxy credentials into the environment for the script.

In [ ]:
import os
from google.colab import ai
MODEL = "google/gemini-3.1-pro-preview"   # fallback: "google/gemini-2.5-pro"
print(ai.generate_text("Reply with the single word OK.", model_name=MODEL))
assert os.environ.get("MODEL_PROXY_HOST"), "MODEL_PROXY_HOST not set"
assert os.environ.get("MODEL_PROXY_API_KEY"), "MODEL_PROXY_API_KEY not set"

## Stage 1: scenarios + frozen test set

In [ ]:
!python draft_inquire_dataset.py scenarios --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}"

In [ ]:
import json
frozen = json.load(open(f"{OUT}/scenarios.json", encoding="utf-8"))
for s in frozen["scenarios"]:
    mark = "TEST" if s["scenario_id"] in frozen["test_scenario_ids"] else "    "
    print(mark, s["scenario_id"], s["risk_category"].ljust(18), s["context"])
print("near-duplicates to review:", frozen["near_duplicates_for_review"])

## Stage 2: previous educator message + 5 child replies per scenario

In [ ]:
!python draft_inquire_dataset.py replies --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}"

## Stage 3: 2 educator drafts per row (150 rows → 300 drafts)

In [ ]:
!python draft_inquire_dataset.py drafts --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}"

In [ ]:
rows = [json.loads(l) for l in open(f"{OUT}/inquire_drafts.jsonl", encoding="utf-8")]
print(len(rows), "rows;", sum(len(r["drafts"]) == 2 for r in rows), "complete;",
      sum(r["split"] == "test" for r in rows), "test rows")
for r in rows[:5]:
    print("
", r["example_id"], r["reply_type"], "|", r["prompt"][0]["content"].rsplit("
", 1)[-1])
    for i, dft in enumerate(r["drafts"]):
        print(f"   draft {i}: {dft}")

Download everything (also kept in Drive).

In [ ]:
import shutil
shutil.make_archive("/content/c3_inquire_drafts", "zip", OUT)
files.download("/content/c3_inquire_drafts.zip")